In [7]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project")  # adjust if your notebook is nested differently
from functions_v2 import *

In [9]:
import numpy as np
import networkx as nx

In [11]:
# 1. Load
edges, n_vertices, edge_weights = load_graph(r"C:\Users\Rakesh\Documents\summer-project\data\raw\bio-CE-GN.edges")
edge_weights = None
print(f"Raw: n_vertices={n_vertices}, edges={len(edges)}")

# 2. Filter to largest connected component
G_nx = nx.Graph()
G_nx.add_edges_from(edges)
num_components = nx.number_connected_components(G_nx)
print(f"Components: {num_components}")

if num_components > 1:
    largest_cc = max(nx.connected_components(G_nx), key=len)
    G_sub = G_nx.subgraph(largest_cc).copy()
    all_nodes = sorted(largest_cc)
    node_map = {old: new for new, old in enumerate(all_nodes)}
    edges = [(node_map[i], node_map[j]) for i, j in G_sub.edges()]
    n_vertices = len(all_nodes)
    G_nx = nx.Graph()
    G_nx.add_edges_from(edges)

print(f"After cleanup: n_vertices={n_vertices}, edges={len(edges)}")

# 3. Phase 1
A, D, L = build_graph_matrices(edges, n_vertices)

# 4. Phase 2
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)
print("L_sigma type:", type(L_sigma))

✓ Loaded: C:\Users\Rakesh\Documents\summer-project\data\raw\bio-CE-GN.edges
  Vertices : 2220
  Edges    : 53683
  Weighted : yes

Raw: n_vertices=2220, edges=53683
Components: 3
After cleanup: n_vertices=2215, edges=53680
✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 2215 x 2215
  Degree range: [1, 242]
  Non-zeros in L: 109575

✓ lambda_min = 0.136723
  (eigenvalues found: [0.         0.13672322])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse

L_sigma type: <class 'scipy.sparse._csr.csr_matrix'>


In [13]:
from networkx.algorithms.community import greedy_modularity_communities, modularity

communities = list(greedy_modularity_communities(G_nx))
communities_sorted = sorted(communities, key=len, reverse=True)

print(f"Found {len(communities_sorted)} communities")
print(f"Sizes: {[len(c) for c in communities_sorted]}")

mod_score = modularity(G_nx, communities_sorted)
print(f"Modularity: {mod_score:.4f}")

gamma_in = list(communities_sorted[0])   # largest community
gamma_out = list(communities_sorted[1])  # second-largest community

print(f"\ngamma_in size: {len(gamma_in)}")
print(f"gamma_out size: {len(gamma_out)}")

overlap = set(gamma_in) & set(gamma_out)
print(f"Overlap (should be empty): {overlap}")

# the key check we now always do upfront - boundary fraction
boundary_fraction = (len(gamma_in) + len(gamma_out)) / n_vertices
print(f"Boundary fraction of total graph: {boundary_fraction:.4f}")

Found 16 communities
Sizes: [816, 736, 474, 56, 35, 29, 20, 15, 8, 7, 5, 4, 3, 3, 2, 2]
Modularity: 0.2638

gamma_in size: 816
gamma_out size: 736
Overlap (should be empty): set()
Boundary fraction of total graph: 0.7007
